In [31]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST00001,Male,0.0,No,Yes,3.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,68.61,205.83,Yes
1,CUST00002,Male,1.0,Yes,No,2.0,Yes,Yes,DSL,No,...,No internet service,Yes,No internet service,No,One year,Yes,Bank transfer (automatic),23.15,46.30,No
2,CUST00003,Female,0.0,No,No,42.0,Yes,Yes,DSL,No,...,No,No internet service,Yes,Yes,Month-to-month,No,Electronic check,42.63,1790.46,Yes
3,CUST00004,Female,0.0,No,Yes,40.0,Yes,Yes,Fiber optic,No,...,Yes,No,No,No internet service,Month-to-month,No,Electronic check,75.04,3001.60,No
4,CUST00005,Male,1.0,Yes,Yes,17.0,Yes,No,Fiber optic,Yes,...,Yes,No,No internet service,No,Two year,Yes,Electronic check,22.38,380.46,Yes


In [32]:
df_fe = df.copy()

if "customerID" in df_fe.columns:
    df_fe = df_fe.drop(columns=["customerID"])

print("Remaining columns after dropping ID columns:")
print(df_fe.columns.tolist())

Remaining columns after dropping ID columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [33]:
# Tenure groups
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"],
    include_lowest=True
)

In [34]:
#Monthly charge groups

df_fe["monthly_charge_group"] = pd.qcut(
    df_fe["MonthlyCharges"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

In [35]:
#Create average charge per month
df_fe["avg_charge_per_month"] = df_fe["TotalCharges"] / (df_fe["tenure"] + 1)

print("Summary of avg_charge_per_month:")
display(df_fe["avg_charge_per_month"].describe())

Summary of avg_charge_per_month:


count    61432.000000
mean        54.744818
std        102.942586
min          0.000000
25%         26.276101
50%         37.351714
75%         56.820469
max       1472.494366
Name: avg_charge_per_month, dtype: float64

In [36]:
#Create high-value customer flag
monthly_median = df_fe["MonthlyCharges"].median()
df_fe["high_value_customer"] = (df_fe["MonthlyCharges"] > monthly_median).astype(int)

print("High-value customer distribution:")
display(df_fe["high_value_customer"].value_counts())

High-value customer distribution:


high_value_customer
0    30721
1    30711
Name: count, dtype: int64

In [37]:

#Create long-term customer flag
df_fe["long_term_customer"] = (df_fe["tenure"] >= 24).astype(int)

print("Long-term customer distribution:")
display(df_fe["long_term_customer"].value_counts())

Long-term customer distribution:


long_term_customer
0    35747
1    25685
Name: count, dtype: int64

In [38]:
#Create service count feature
service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df_fe["service_count"] = df_fe[service_cols].apply(
    lambda row: sum(row == "Yes"),
    axis=1
)

print("Service count summary:")
display(df_fe["service_count"].describe())

Service count summary:


count    61432.000000
mean         0.965035
std          0.996722
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max          6.000000
Name: service_count, dtype: float64

In [39]:
#Create support-services flag
df_fe["has_support_services"] = (
    (df_fe["OnlineSecurity"] == "Yes") |
    (df_fe["TechSupport"] == "Yes")
).astype(int)

print("Support services flag distribution:")
display(df_fe["has_support_services"].value_counts())

Support services flag distribution:


has_support_services
0    43670
1    17762
Name: count, dtype: int64

In [40]:
#Create month-to-month contract flag
df_fe["is_month_to_month"] = (df_fe["Contract"] == "Month-to-month").astype(int)

print("Month-to-month flag distribution:")
display(df_fe["is_month_to_month"].value_counts())

Month-to-month flag distribution:


is_month_to_month
1    36857
0    24575
Name: count, dtype: int64

In [41]:
#Create automatic payment flag
df_fe["uses_auto_payment"] = df_fe["PaymentMethod"].isin([
    "Bank transfer (automatic)",
    "Credit card (automatic)"
]).astype(int)

print("Automatic payment flag distribution:")
display(df_fe["uses_auto_payment"].value_counts())

Automatic payment flag distribution:


uses_auto_payment
0    39835
1    21597
Name: count, dtype: int64

In [42]:
#Convert binary yes/no columns into 0/1
binary_map = {"Yes": 1, "No": 0}

binary_cols = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]

for col in binary_cols:
    if col in df_fe.columns:
        df_fe[col] = df_fe[col].map(binary_map)

print("Preview after binary encoding:")
display(df_fe[binary_cols].head())

Preview after binary encoding:


,Partner,Dependents,PhoneService,PaperlessBilling,Churn
0,0,1,1,0,1
1,1,0,1,1,0
2,0,0,1,0,1
3,0,1,1,0,0
4,1,1,1,1,1


In [43]:
#Encode gender
if "gender" in df_fe.columns:
    df_fe["gender"] = df_fe["gender"].map({"Male": 1, "Female": 0})

print("Gender value counts after encoding:")
display(df_fe["gender"].value_counts(dropna=False))

Gender value counts after encoding:


gender
0    30877
1    30555
Name: count, dtype: int64

In [44]:
#Ensure SeniorCitizen is numeric
if "SeniorCitizen" in df_fe.columns:
    df_fe["SeniorCitizen"] = pd.to_numeric(df_fe["SeniorCitizen"], errors="coerce")

print("SeniorCitizen values:")
display(df_fe["SeniorCitizen"].value_counts(dropna=False))

SeniorCitizen values:


SeniorCitizen
0.0    49082
1.0    12350
Name: count, dtype: int64

In [45]:

categorical_to_encode = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
    "tenure_group",
    "monthly_charge_group"
]

# Keep only columns that actually exist
categorical_to_encode = [col for col in categorical_to_encode if col in df_fe.columns]

df_model = pd.get_dummies(df_fe, columns=categorical_to_encode, drop_first=True)

print("Shape after one-hot encoding:")
print(df_model.shape)

print("\nPreview of encoded dataset:")
display(df_model.head())

Shape after one-hot encoding:
(61432, 42)

Preview of encoded dataset:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,PaymentMethod_Unknown,monthly_charge_group_Medium,monthly_charge_group_High,monthly_charge_group_Very High
0,1,0.0,0,1,3.0,1,0,68.61,205.83,1,...,False,False,False,False,False,True,False,False,False,True
1,1,1.0,1,0,2.0,1,1,23.15,46.30,0,...,False,True,False,False,False,False,False,False,False,False
2,0,0.0,0,0,42.0,1,0,42.63,1790.46,1,...,True,False,False,False,True,False,False,False,True,False
3,0,0.0,0,1,40.0,1,0,75.04,3001.60,0,...,False,False,False,False,True,False,False,False,False,True
4,1,1.0,1,1,17.0,1,1,22.38,380.46,1,...,False,False,True,False,True,False,False,False,False,False


In [46]:
#Check missing values after feature engineering
missing_after_fe = df_model.isna().sum().sort_values(ascending=False)
missing_after_fe = missing_after_fe[missing_after_fe > 0]

print("Remaining missing values:")
display(missing_after_fe)

Remaining missing values:


Series([], dtype: int64)

In [47]:
#Check final data types
print("Final data types:")
display(df_model.dtypes)

Final data types:


gender                                     int64
SeniorCitizen                            float64
Partner                                    int64
Dependents                                 int64
tenure                                   float64
PhoneService                               int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
avg_charge_per_month                     float64
high_value_customer                        int32
long_term_customer                         int32
service_count                              int64
has_support_services                       int32
is_month_to_month                          int32
uses_auto_payment                          int32
MultipleLines_No phone service              bool
MultipleLines_Yes                           bool
InternetService_Fiber optic                 bool
InternetService_No  

In [48]:

#Save the final featured dataset
df_model.to_csv("../data/processed/churn_featured.csv", index=False)

print("Feature-engineered dataset saved successfully.")

Feature-engineered dataset saved successfully.
